# Stage 1 — Baseline Dense RAG
`Query → Dense → Top-k → Base LLM`

## 0. GPU

In [1]:
import os, gc, torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"
print('Input:', os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'YOK')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} | {p.total_memory/1024**3:.1f} GB')
else:
    print('GPU bulunamadi — Kaggle > Settings > Accelerator > T4 x2')

Input: ['datasets']
GPU 0: Tesla T4 | 14.6 GB
GPU 1: Tesla T4 | 14.6 GB


## 1. Kurulum

In [2]:
!pip install -q sentence-transformers==3.4.1 faiss-cpu rouge-score nltk pandas tqdm
!pip install -U bitsandbytes
!pip install -q accelerate
import nltk; nltk.download('punkt', quiet=True)
print('Kurulum tamam.')

Kurulum tamam.


## 2. Config

In [3]:
import os, json, re, math, gc, pickle
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict

@dataclass
class Config:
    SYSTEM_NAME  : str  = 'System1_Baseline'
    SYSTEM_DESC  : str  = 'Base BGE-M3 + Dense Retrieval + Base Qwen2.5-7B'
    DRIVE_DIR    : str  = '/kaggle/input/datasets/ardayildiz29/legalo'
    CORPUS_FILE  : str  = 'corpus.jsonl'
    GOLD_FILE    : str  = 'gold_benchmark.json'
    INDEX_DIR    : str  = '/kaggle/working/legal_rag/index_s1'
    MODEL_DIR    : str  = '/kaggle/working/legal_rag/models_s1'
    RESULTS_DIR  : str  = '/kaggle/working/legal_rag/results'
    EMBED_BASE   : str  = 'BAAI/bge-m3'
    EMBED_DEVICE : str  = 'cuda'
    EMBED_BATCH  : int  = 64
    TOP_K_DENSE  : int  = 20
    TOP_K_RERANK_IN : int = 20
    TOP_K_FINAL  : int  = 5
    LLM_MODEL    : str  = 'Qwen/Qwen2.5-7B-Instruct'
    MAX_NEW_TOKENS: int = 128
    BENCH_SIZE   : int  = 240
    RANDOM_SEED  : int  = 42

CFG = Config()
for d in [CFG.INDEX_DIR, CFG.MODEL_DIR, CFG.RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)
with open(f'{CFG.RESULTS_DIR}/{CFG.SYSTEM_NAME}_config.json', 'w', encoding='utf-8') as f:
    json.dump(asdict(CFG), f, ensure_ascii=False, indent=2)
print(f'Sistem   : {CFG.SYSTEM_NAME}')
print(f'Aciklama : {CFG.SYSTEM_DESC}')
print('Config kaydedildi.')

Sistem   : System1_Baseline
Aciklama : Base BGE-M3 + Dense Retrieval + Base Qwen2.5-7B
Config kaydedildi.


## 3. Veri

In [4]:
import random
from tqdm import tqdm

DRIVE = Path(CFG.DRIVE_DIR)
for name, path in [('corpus', DRIVE/CFG.CORPUS_FILE), ('gold', DRIVE/CFG.GOLD_FILE)]:
    if not path.exists():
        raise FileNotFoundError(f'Eksik: {path}')
    print(f'  {name}: OK ({path.stat().st_size/1024**2:.1f} MB)')

def load_jsonl(p):
    rows = []
    with open(p, encoding='utf-8') as f:
        for ln, line in enumerate(f,1):
            line = line.strip()
            if line:
                try: rows.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f'{p.name} satir={ln}: {e}')
    return rows

print('Corpus yukleniyor...')
CORPUS: List[Dict] = [{'id':r['id'],'text':r['text'].strip(),
                        'title':r.get('title',''),'metadata':r.get('metadata',{})}
                       for r in load_jsonl(DRIVE/CFG.CORPUS_FILE)]
print(f'Corpus: {len(CORPUS)} chunk')
corpus_id_set = {c['id'] for c in CORPUS}
assert len(corpus_id_set) == len(CORPUS), 'Duplicate corpus id!'

print('Benchmark yukleniyor...')
with open(DRIVE/CFG.GOLD_FILE, encoding='utf-8') as f:
    gold_raw = json.load(f)
BENCHMARK = []
for item in gold_raw:
    rel_ids = [s.get('source_id') or s.get('corpus_row_id')
               for s in item.get('gold_sources',[]) if s.get('source_id') or s.get('corpus_row_id')]
    q = item.get('question','').strip(); a = item.get('verified_answer','').strip()
    if q and a and rel_ids:
        BENCHMARK.append({'question_id':item.get('question_id'),'question':q,'answer':a,'relevant_ids':rel_ids})
random.seed(CFG.RANDOM_SEED); random.shuffle(BENCHMARK)
BENCHMARK = BENCHMARK[:CFG.BENCH_SIZE]
covered = sum(1 for b in BENCHMARK if any(r in corpus_id_set for r in b['relevant_ids']))
print(f'Benchmark: {len(BENCHMARK)} soru | Coverage: {covered}/{len(BENCHMARK)}')

  corpus: OK (22.4 MB)
  gold: OK (0.8 MB)
Corpus yukleniyor...
Corpus: 7643 chunk
Benchmark yukleniyor...
Benchmark: 240 soru | Coverage: 240/240


## 4. FAISS (Base BGE-M3)

In [5]:
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

EMBED_PATH = "BAAI/bge-m3"
INDEX_PATH = Path(CFG.INDEX_DIR) / 'faiss_s1.bin'
DOCS_PATH  = Path(CFG.INDEX_DIR) / 'docs_s1.pkl'

rebuild = True
if INDEX_PATH.exists() and DOCS_PATH.exists():
    try:
        _idx = faiss.read_index(str(INDEX_PATH))
        with open(DOCS_PATH,'rb') as f: _docs = pickle.load(f)
        if _idx.ntotal == len(CORPUS):
            FAISS_INDEX = _idx; rebuild = False
            print(f'FAISS cache: {FAISS_INDEX.ntotal} vektor (Base BGE-M3)')
        else:
            print('Cache uyusmuyor; yeniden olusturuluyor.')
    except Exception as e:
        print(f'Cache hata: {e}')

if rebuild:
    print(f'Encoder yukleniyor (Base BGE-M3)...')
    _enc  = SentenceTransformer(EMBED_PATH, device=CFG.EMBED_DEVICE)
    texts = [c['text'] for c in CORPUS]
    print(f'{len(texts)} chunk embed ediliyor...')
    embs  = _enc.encode(texts, batch_size=CFG.EMBED_BATCH, normalize_embeddings=True,
                        show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    dim = embs.shape[1]
    FAISS_INDEX = faiss.IndexFlatIP(dim)
    FAISS_INDEX.add(embs)
    faiss.write_index(FAISS_INDEX, str(INDEX_PATH))
    with open(DOCS_PATH,'wb') as f: pickle.dump(CORPUS, f)
    del _enc, embs; gc.collect()
    print(f'FAISS hazir: {FAISS_INDEX.ntotal} vektor, dim={dim}')

print(f'Query encoder yukleniyor (Base BGE-M3)...')
QUERY_ENCODER = SentenceTransformer(EMBED_PATH, device=CFG.EMBED_DEVICE)
print('Hazir.')

2026-05-31 09:34:37.957768: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780220077.985330     260 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780220077.994869     260 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780220078.032964     260 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780220078.032990     260 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780220078.032993     260 computation_placer.cc:177] computation placer alr

FAISS cache: 7643 vektor (Base BGE-M3)
Query encoder yukleniyor (Base BGE-M3)...
Hazir.


## 5. LLM (Base Qwen)

In [6]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

gc.collect(); torch.cuda.empty_cache()
print(f'Boş VRAM: {torch.cuda.mem_get_info()[0]/1024**3:.2f} GB')

print('Base Qwen2.5-7B yukleniyor (4-bit)...')
bnb_inf = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16
)
LLM_MODEL_OBJ = AutoModelForCausalLM.from_pretrained(
    CFG.LLM_MODEL, quantization_config=bnb_inf, device_map='auto',
    trust_remote_code=True, low_cpu_mem_usage=True,
)
LLM_MODEL_OBJ.eval()
LLM_TOKENIZER = AutoTokenizer.from_pretrained(CFG.LLM_MODEL, trust_remote_code=True)
if LLM_TOKENIZER.pad_token is None:
    LLM_TOKENIZER.pad_token = LLM_TOKENIZER.eos_token
LLM_TOKENIZER.padding_side = 'left'
print(f'Qwen hazir | VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB')


Boş VRAM: 12.33 GB
Base Qwen2.5-7B yukleniyor (4-bit)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen hazir | VRAM: 3.9 GB


## 6. Pipeline

In [7]:
import warnings, numpy as np
warnings.filterwarnings('ignore')

SYSTEM_PROMPT_INF = (
    'Sen bir Turk hukuku RAG asistanisin.\n'
    'Yalnizca kullanicinin verdigi kaynak metne dayanarak cevap ver.\n'
    'Kaynakta olmayan bilgiyi uretme.\n'
    'Cevabinin sonunda kaynak/citation bilgisini mutlaka belirt.'
)

def chunk_citation(c):
    meta = c.get('metadata', {})
    return meta.get('citation_label') or meta.get('chunk_id') or c.get('id','Turk Hukuku')

def build_prompt(query, chunks):
    context = ''
    for i, c in enumerate(chunks, 1):
        meta = c.get('metadata', {}); title = c.get('title') or meta.get('category','Turk Hukuku')
        context += f'[Belge {i} | {title} | Kaynak: {chunk_citation(c)}]\n{c["text"]}\n\n'
    return SYSTEM_PROMPT_INF, f'{context}Soru: {query}\n\nKisa ve kaynakli yanit (2-4 cumle):'

def generate(query, chunks):
    sys_p, user_p = build_prompt(query, chunks)
    msgs = [{'role':'system','content':sys_p},{'role':'user','content':user_p}]
    text = LLM_TOKENIZER.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = LLM_TOKENIZER([text], return_tensors='pt').to(LLM_MODEL_OBJ.device)
    with torch.no_grad(), torch.cuda.amp.autocast():
        out = LLM_MODEL_OBJ.generate(
            **inputs,
            max_new_tokens=CFG.MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=LLM_TOKENIZER.eos_token_id,
            repetition_penalty=1.1,
            use_cache=True,
        )
    return LLM_TOKENIZER.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def dense_retrieve(query, top_k):
    q_emb = QUERY_ENCODER.encode([query], normalize_embeddings=True,
                                  convert_to_numpy=True).astype(np.float32)
    scores, idxs = FAISS_INDEX.search(q_emb, top_k)
    return [dict(**CORPUS[i], dense_score=float(s)) for s, i in zip(scores[0], idxs[0]) if i >= 0]

def rag_pipeline(question, verbose=False):
    candidates   = dense_retrieve(question, CFG.TOP_K_RERANK_IN)
    all_ids      = [c['id'] for c in candidates]
    final_chunks = candidates[:CFG.TOP_K_FINAL]
    answer       = generate(question, final_chunks)
    sources      = list({c.get('title','Turk Hukuku') for c in final_chunks})
    if verbose:
        print(f'Soru: {question}')
        [print(f'  [{i+1}] score={c.get("dense_score",0):.4f} | {c["text"][:80]}...') for i,c in enumerate(final_chunks)]
        print(f'Cevap: {answer[:300]}')
    return {'question':question,'answer':answer,'retrieved_chunks':final_chunks,'all_retrieved':all_ids,'sources':sources}

print('Pipeline testi (dense-only)...')
_ = rag_pipeline('Susma hakki nedir?', verbose=True)

Pipeline testi (dense-only)...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Soru: Susma hakki nedir?
  [1] score=0.5067 | Habeas Corpus, savunma hakkı ve tutukluluk durumu konusunda özgürlüklerin korunm...
  [2] score=0.5041 | Susma hakkı, kişinin kendi lehine veya aleyhine ifade verme kararını özgürce kul...
  [3] score=0.4791 | Yakalanan veya gözaltına alınan bir kişinin susma hakkı, kişinin kendisine yönel...
  [4] score=0.4616 | Sanığın susma hakkı, sanığın kendini ifade etme zorunluluğu olmadan sessiz kalma...
  [5] score=0.4261 | Bir davada mahkemenin veya tarafların yapmış oldukları bir usul işlemi nedeniyle...
Cevap: Susma hakkı, şüpheli veya sanığın kendi lehine veya aleyhine ifade vermeyi seçebilmelerine olanak tanıyan temel bir hukuki hakkıdır. Bu hakkı, anayasal ve hukuki temelde korunan, yakalanan veya gözaltına alınan kişilerin kimlik bilgisi dışında cevap verme zorunluluğundan kurtulmasına ve sessiz kalma


## 7. Metrikler

In [8]:
from rouge_score import rouge_scorer as rs_module
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd

def tokenize_tr(text):
    return [t for t in re.sub(r'[^\w\s]',' ',text.lower()).split() if len(t)>1]

def recall_at_k(ret, rel, k):
    if not rel: return None
    return round(len(set(ret[:k])&set(rel))/len(rel), 4)

def mrr_score(ret, rel):
    if not rel: return None
    for rank, did in enumerate(ret, 1):
        if did in set(rel): return round(1.0/rank, 4)
    return 0.0

def ndcg_at_k(ret, rel, k):
    if not rel: return None
    rel_set = set(rel)
    dcg  = sum(1/math.log2(i+2) for i,d in enumerate(ret[:k]) if d in rel_set)
    idcg = sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return round(dcg/idcg, 4) if idcg>0 else 0.0

def f1_token(pred, gt):
    p = set(tokenize_tr(pred)); g = set(tokenize_tr(gt))
    if not p or not g: return 0.0
    c = p&g
    if not c: return 0.0
    return round(2*len(c)/(len(p)+len(g)), 4)

def bleu_score(pred, gt):
    return round(sentence_bleu([tokenize_tr(gt)], tokenize_tr(pred),
                 smoothing_function=SmoothingFunction().method1), 4)

def rouge_scores(pred, gt):
    s = rs_module.RougeScorer(['rouge1','rouge2','rougeL']).score(gt, pred)
    return {k: round(s[k].fmeasure, 4) for k in s}

def faithfulness_score(answer, chunks):
    ctx = ' '.join(c['text'] for c in chunks)
    a_t = set(tokenize_tr(answer)); c_t = set(tokenize_tr(ctx))
    return round(len(a_t&c_t)/len(a_t), 4) if a_t else 0.0

def evaluate(benchmark, desc=''):
    rows = []
    for item in tqdm(benchmark, desc=desc or 'Evaluating'):
        result  = rag_pipeline(item['question'])
        ret_ids = [c['id'] for c in result['retrieved_chunks']]
        all_ids = result.get('all_retrieved', ret_ids)
        rel     = item['relevant_ids']; gt = item['answer']
        rg = rouge_scores(result['answer'], gt)
        rows.append({
            'question_id': item.get('question_id'),
            'question'   : item['question'][:80],
            'recall@5'   : recall_at_k(all_ids,rel,5),
            'recall@10'  : recall_at_k(all_ids,rel,10),
            'mrr'        : mrr_score(all_ids,rel),
            'ndcg@5'     : ndcg_at_k(all_ids,rel,5),
            'f1'         : f1_token(result['answer'],gt),
            'bleu'       : bleu_score(result['answer'],gt),
            'rouge1'     : rg['rouge1'], 'rouge2': rg['rouge2'], 'rougeL': rg['rougeL'],
            'faithfulness': faithfulness_score(result['answer'], result['retrieved_chunks']),
            'answer'     : result['answer'][:200],
            'gold_answer': gt[:200],
        })
    df = pd.DataFrame(rows)
    summary = {m: round(df[m].dropna().mean(),4) for m in
               ['recall@5','recall@10','mrr','ndcg@5','f1','bleu','rouge1','rouge2','rougeL','faithfulness']}
    return {'details':df, 'summary':summary}

## 8. Evaluate

In [9]:
print(f'=== {CFG.SYSTEM_NAME} --- {len(BENCHMARK)} soru ===')
EVAL_OUT = evaluate(BENCHMARK, desc=CFG.SYSTEM_NAME)
line = '='*64
print(f'\n{line}\n  {CFG.SYSTEM_NAME}\n  {CFG.SYSTEM_DESC}\n{line}')
print('  RETRIEVAL METRİKLERİ')
for m in ['recall@5','recall@10','mrr','ndcg@5']:
    print(f'    {m:<15s}: {EVAL_OUT["summary"].get(m,"N/A")}')
print('  QA METRİKLERİ')
for m in ['f1','bleu','rouge1','rouge2','rougeL','faithfulness']:
    print(f'    {m:<15s}: {EVAL_OUT["summary"].get(m,"N/A")}')
print(line)
hall = EVAL_OUT['details'][EVAL_OUT['details']['faithfulness'] < 0.3]
print(f'\nHallucination suphelisi (faithfulness<0.3): {len(hall)}/{len(BENCHMARK)}')

=== System1_Baseline --- 240 soru ===


System1_Baseline: 100%|██████████| 240/240 [1:10:54<00:00, 17.73s/it]


  System1_Baseline
  Base BGE-M3 + Dense Retrieval + Base Qwen2.5-7B
  RETRIEVAL METRİKLERİ
    recall@5       : 0.7875
    recall@10      : 0.825
    mrr            : 0.6826
    ndcg@5         : 0.7044
  QA METRİKLERİ
    f1             : 0.3739
    bleu           : 0.1104
    rouge1         : 0.3795
    rouge2         : 0.2447
    rougeL         : 0.2903
    faithfulness   : 0.6288

Hallucination suphelisi (faithfulness<0.3): 4/240


## 9. Kaydet

In [10]:
from datetime import datetime
ts       = datetime.now().strftime('%Y%m%d_%H%M%S')
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU'
result_path = f'{CFG.RESULTS_DIR}/{CFG.SYSTEM_NAME}_{ts}.json'
with open(result_path,'w',encoding='utf-8') as f:
    json.dump({'system':CFG.SYSTEM_NAME,'desc':CFG.SYSTEM_DESC,'timestamp':ts,
               'gpu':gpu_name,'benchmark':len(BENCHMARK),'config':asdict(CFG),
               'metrics':EVAL_OUT['summary']}, f, ensure_ascii=False, indent=2)
csv_path = f'{CFG.RESULTS_DIR}/{CFG.SYSTEM_NAME}_details_{ts}.csv'
EVAL_OUT['details'].to_csv(csv_path, index=False, encoding='utf-8')
print(f'Kaydedildi:')
print(f'  Ozet : {result_path}')
print(f'  Detay: {csv_path}')
print(f'\n OK {CFG.SYSTEM_NAME} tamamlandi!')

Kaydedildi:
  Ozet : /kaggle/working/legal_rag/results/System1_Baseline_20260531_104651.json
  Detay: /kaggle/working/legal_rag/results/System1_Baseline_details_20260531_104651.csv

 OK System1_Baseline tamamlandi!


## Interactive

In [11]:
question = "SORUNUZU YAZIN :)"
if question and question != "SORUNUZU YAZIN :)":
    result = rag_pipeline(question, verbose=True)
    print('\n' + '='*60)
    print('CEVAP:', result['answer'])